   
Databricks notebook source
SENSE 프로젝트 — Gold Layer: silver → gold
담당: 1조 (정형 데이터)

목적:
  yfinance / FRED / FX / semiconductor / kfinance silver 를
  date 기준으로 통합하여 ML 학습용 Gold 피처 마트를 생성하고
  ADLS Gold 레이어에 저장

실행 순서:
  01_yfinance → 02_fred → 03_fx → 04_semiconductor → 05_kfinance
  → ✅ 이 노트북

[JOIN 전략]
  yfinance silver (effective_kr_date) 를 기준 캘린더로 사용
  FRED / FX / semiconductor / kfinance silver 를 LEFT JOIN
  → 뉴스 팀 silver 완성 시 LEFT JOIN 추가만 하면 됨

[Gold 전용 파생열]
  risk_off_composite  : FX 급등 + FRED 금리차 역전 복합 신호
  macro_stress_score  : 매크로 종합 스트레스 점수
  fear_composite      : 옵션 IV + 환율 + 금리 복합 공포 지수
  semi_risk_signal    : 수출 감소 + 변동성 고조 동시 발생
  korea_sensitivity   : 환율 × IV 한국 시장 민감도
  global_risk_regime  : 다중 신호 기반 위험 국면 (0/1/2)

[타겟 변수]
  is_high_risk : volatility_5d 가 상위 25% 초과 시 1 (하방 리스크 라벨)

# 0. 스토리지 계정 설정 및 ADLS OAuth 인증


In [0]:
# ============================================================
# 스토리지 계정 및 경로 상수 정의
# ============================================================
STORAGE_ACCOUNT  = "3dtteam1adls"
SECRET_SCOPE     = "sense-kv"
BRONZE_CONTAINER    = "raw"
SILVER_CONTAINER = "curated"
GOLD_CONTAINER = "feature"

BASE_PATH_BRONZE    = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_SILVER = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_GOLD = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Silver 경로
SILVER_YFINANCE  = f"{BASE_PATH_SILVER}/yfinance"
SILVER_FRED      = f"{BASE_PATH_SILVER}/fred"
SILVER_FX        = f"{BASE_PATH_SILVER}/fx"
SILVER_SEMI      = f"{BASE_PATH_SILVER}/semiconductor"
SILVER_KFINANCE  = f"{BASE_PATH_SILVER}/kfinance"

# Gold 저장 경로
GOLD_PATH       = f"{BASE_PATH_GOLD}/sense_macro"

# master_calendar 경로
CALENDAR_PATH   = f"{BASE_PATH_SILVER}/master_calendar.parquet"

# ============================================================
# ADLS Gen2 OAuth 인증 (Service Principal)
# ============================================================
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=SECRET_SCOPE, key='adls-tenant-id')}/oauth2/token"
)

print(f"✅ ADLS OAuth 인증 설정 완료: {STORAGE_ACCOUNT}")

# 1. master_calendar 로드

Gold JOIN 의 기준 날짜를 **한국 영업일** 로 통일합니다.
yfinance 의 `effective_kr_date` 가 이미 한국 영업일 기준이므로
캘린더는 날짜 범위 필터 및 최종 검증 용도로 사용합니다.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date
from pyspark.sql import Window

calendar_df = (
    spark.read
    .parquet(CALENDAR_PATH)
    .withColumn("기준일자", to_date(col("기준일자"), "yyyy-MM-dd"))
    .select("기준일자", "한국_휴장일_여부", "미국_휴장일_여부")
)

# 양국 공통 영업일 (Gold JOIN 최종 기준)
common_biz_days = (
    calendar_df
    .filter(
        (col("한국_휴장일_여부") == False) &
        (col("미국_휴장일_여부") == False)
    )
    .select(col("기준일자").alias("date"))
)

print(f"✅ master_calendar 로드 완료")
print(f"   양국 공통 영업일: {common_biz_days.count()}일")


   
# 2. Silver 데이터 로드

Silver 5개를 각각 로드합니다.

| Silver | 날짜 컬럼 | 주기 | 주요 컬럼 |
|---|---|---|---|
| yfinance | `effective_kr_date` | 일별 | Close, log_return, volatility_5d |
| FRED | `date` | 일별 | DGS10, DGS2, yield_spread, stagnation_pressure |
| FX | `date` | 일별 | usd_krw, usd_krw_pct, risk_off_flag |
| semiconductor | `date` | **일별 (FF완료)** | export_usd, export_change_pct, export_momentum |
| kfinance | `date` | **월별 28행** | avg_iv, call_volume, call_oi, iv_surge_flag |

In [0]:
# ── yfinance silver 로드 ────────────────────────────────────────────
yf_silver = spark.read.parquet(SILVER_YFINANCE)

print(f"✅ yfinance silver: {yf_silver.count():,}행")
print(f"   컬럼: {yf_silver.columns}")
display(yf_silver.limit(5))


In [0]:
# ── FRED silver 로드 ────────────────────────────────────────────────
fred_silver = spark.read.parquet(SILVER_FRED)

print(f"✅ FRED silver: {fred_silver.count():,}행")
print(f"   컬럼: {fred_silver.columns}")
display(fred_silver.limit(5))


In [0]:
# ── FX silver 로드 ──────────────────────────────────────────────────
fx_silver = spark.read.parquet(SILVER_FX)

print(f"✅ FX silver: {fx_silver.count():,}행")
print(f"   컬럼: {fx_silver.columns}")
display(fx_silver.limit(5))


In [0]:
# ── Semiconductor silver 로드 ──────────────────────────────────
semi_silver = spark.read.parquet(SILVER_SEMI)

print(f"✅ Semiconductor silver: {semi_silver.count():,}행")
print(f"   컬럼: {semi_silver.columns}")
display(semi_silver.limit(5))

In [0]:
# ── KFinance (KOSPI 옵션) silver 로드 ───────────────────────
kfin_silver = spark.read.parquet(SILVER_KFINANCE)

print(f"✅ KFinance silver: {kfin_silver.count():,}행")
print(f"   컬럼: {kfin_silver.columns}")
display(kfin_silver.limit(5))

# 3. yfinance Wide 포맷 변환

yfinance silver 는 **Long 포맷** (한 행 = 한 티커 × 한 날짜) 입니다.
Gold 에서 ML 모델에 바로 넣으려면 **Wide 포맷** (한 행 = 한 날짜) 으로 변환해야 합니다.

```
[Long 포맷]                    [Wide 포맷]
date   | Ticker | Close        date   | NVDA_close | TSM_close | ^SOX_close ...
-------|--------|------        -------|------------|-----------|------------
04-09  | NVDA   | 112.3        04-09  | 112.3      | 145.6     | 4230.4
04-09  | TSM    | 145.6
04-09  | ^SOX   | 4230.4
```

타겟 변수의 기준이 되는 `NVDA` 의 `volatility_5d` 를 대표로 사용합니다.


In [0]:
# ── Gold 에 포함할 티커 및 컬럼 정의 ─────────────────────────
# 반도체 주가 (타겟 변수 기반)
SEMI_TICKERS = ["NVDA", "TSM", "AMD", "INTC", "ASML", "MU", "WDC", "^SOX",
                "005930.KS", "000660.KS"]

# ── 티커별 종가 피벗 ───────────────────────────────────────
yf_close_pivot = (
    yf_silver
    .filter(col("Ticker").isin(SEMI_TICKERS))
    .groupBy("effective_kr_date")
    .pivot("Ticker", SEMI_TICKERS)
    .agg(F.first("Close"))
    .withColumnRenamed("effective_kr_date", "date")
)

# ── NVDA 기준 volatility_5d, log_return 추출 (타겟 변수용) ────
# ⚠️ F.last() 집계: 한국 연휴 시 복수 US 거래일이 동일 effective_kr_date에
#    매핑되므로, 가장 최근(last) US 거래일 값을 사용 (한국 개장 직전 데이터)
nvda_metrics = (
    yf_silver
    .filter(col("Ticker") == "NVDA")
    .groupBy(col("effective_kr_date").alias("date"))
    .agg(
        F.last("log_return").alias("NVDA_log_return"),
        F.last("volatility_gk").alias("NVDA_volatility_gk"),
        F.last("volatility_5d").alias("NVDA_volatility_5d")
    )
)

# ── SOX 지수 변동성 추출 ─────────────────────────────────
sox_metrics = (
    yf_silver
    .filter(col("Ticker") == "^SOX")
    .groupBy(col("effective_kr_date").alias("date"))
    .agg(
        F.last("log_return").alias("SOX_log_return"),
        F.last("volatility_5d").alias("SOX_volatility_5d")
    )
)

print(f"✅ yfinance Wide 변환 완료")
print(f"   종가 피벗: {yf_close_pivot.count()}행 × {len(yf_close_pivot.columns)}컬럼")
print(f"   NVDA 메트릭: {nvda_metrics.count()}행 (F.last 집계)")
print(f"   SOX  메트릭: {sox_metrics.count()}행 (F.last 집계)")
display(yf_close_pivot.limit(5))

   
# 4. Gold JOIN

`date` 기준으로 데이터를 순서대로 LEFT JOIN 합니다.

```
yf_close_pivot (기준)
    LEFT JOIN nvda_metrics      ON date
    LEFT JOIN sox_metrics       ON date
    LEFT JOIN fred_silver       ON date
    LEFT JOIN fx_silver         ON date
    LEFT JOIN semi_silver       ON date  ← NEW (반도체 수출)
    LEFT JOIN kfin_silver       ON date  ← NEW (코스피 옵션)
```

> **kfinance 주의:** 월별 28행만 존재 → 대부분 날짜에서 null → Forward Fill로 일별 확장
> **semiconductor:** 이미 Forward Fill 적용된 일별 데이터 (511행)

In [0]:
# ── JOIN 실행 ────────────────────────────────────────────────
gold_df = (
    yf_close_pivot
    .join(nvda_metrics,  on="date", how="left")
    .join(sox_metrics,   on="date", how="left")
    .join(
        fred_silver.select(
            "date", "DGS10", "DGS2", "DFF", "DFII10",
            "T10Y2Y", "BAMLH0A0HYM2",
            "yield_spread", "yield_spread_change", "stagnation_pressure"
        ),
        on="date", how="left"
    )
    .join(
        fx_silver.select(
            "date", "usd_krw", "usd_krw_change",
            "usd_krw_pct", "risk_off_flag"
        ),
        on="date", how="left"
    )
    .join(
        semi_silver.select(
            "date", "export_usd", "export_change_pct", "export_momentum"
        ),
        on="date", how="left"
    )
    .join(
        kfin_silver.select(
            "date", "call_volume", "call_oi", "avg_iv",
            "iv_change", "vol_change_pct", "iv_surge_flag"
        ),
        on="date", how="left"
    )
    .orderBy("date")
)

print(f"✅ Gold JOIN 완료")
print(f"   행수: {gold_df.count():,}행")
print(f"   컬럼 수: {len(gold_df.columns)}개")
display(gold_df.limit(5))

# 5. JOIN 후 잔여 결측치 Forward Fill

LEFT JOIN 후 일부 날짜에 FRED / FX 값이 없을 수 있습니다.
(미국·한국 공휴일 불일치 구간)
직전 유효값으로 채웁니다.


In [0]:
w_ffill = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

# yfinance 종가 Forward Fill (US 휴장일 → KR 영업일에 null 발생)
YF_PRICE_COLS = ["NVDA", "TSM", "AMD", "INTC", "ASML", "MU", "WDC",
                 "^SOX", "005930.KS", "000660.KS"]

# yfinance 파생 메트릭 Forward Fill
YF_METRIC_COLS = ["NVDA_log_return", "NVDA_volatility_gk", "NVDA_volatility_5d",
                  "SOX_log_return", "SOX_volatility_5d"]

# FRED 컬럼 Forward Fill
FRED_COLS = ["DGS10", "DGS2", "DFF", "DFII10", "T10Y2Y",
             "BAMLH0A0HYM2", "yield_spread", "yield_spread_change",
             "stagnation_pressure"]

# FX 컬럼 Forward Fill
FX_COLS = ["usd_krw", "usd_krw_change", "usd_krw_pct", "risk_off_flag"]

# Semiconductor 컬럼 Forward Fill (일별 FF완료이지만 JOIN 누락 대비)
SEMI_COLS = ["export_usd", "export_change_pct", "export_momentum"]

# KFinance 컬럼 Forward Fill (월별 28행 → 일별 확장)
KFIN_COLS = ["call_volume", "call_oi", "avg_iv",
             "iv_change", "vol_change_pct", "iv_surge_flag"]

ALL_FFILL_COLS = YF_PRICE_COLS + YF_METRIC_COLS + FRED_COLS + FX_COLS + SEMI_COLS + KFIN_COLS

gold_ffill_df = gold_df
for c in ALL_FFILL_COLS:
    gold_ffill_df = gold_ffill_df.withColumn(
        c,
        F.last(col(f"`{c}`"), ignorenulls=True).over(w_ffill)
    )

print("✅ POST JOIN Forward Fill 완료")
print(f"   Forward Fill 대상: {len(ALL_FFILL_COLS)}컬럼")
print(f"   - yfinance 종가: {len(YF_PRICE_COLS)}컬럼")
print(f"   - yfinance 파생: {len(YF_METRIC_COLS)}컬럼")
print(f"   - FRED/FX/Semi/KFin: {len(FRED_COLS + FX_COLS + SEMI_COLS + KFIN_COLS)}컬럼")

   
# 6. Gold 전용 파생열 생성

### 파생열
| 컬럼명 | 필요 소스 | 의미 |
|---|---|---|
| `risk_off_composite` | FX + FRED | 환율 급등 AND 금리차 역전 동시 발생 |
| `macro_stress_score` | FRED + FX | 매크로 종합 스트레스 점수 |
| `fear_composite` | kfinance + FX + FRED | 복합 공포 지수 (IV + 환율 + 금리 신호 합산) |
| `semi_risk_signal` | semiconductor + yfinance | 수출 감소 + 변동성 고조 동시 발생 |
| `korea_sensitivity` | FX + kfinance | 한국 시장 민감도 (환율 × IV 상호작용) |
| `global_risk_regime` | 전체 | 위험 국면 분류 (0=저위험, 1=중위험, 2=고위험) |

In [0]:
gold_featured_df = (
    gold_ffill_df

    # ── [기존] risk_off_composite ─────────────────────────────
    .withColumn(
        "risk_off_composite",
        F.when(
            (col("risk_off_flag") == 1) & (col("yield_spread") < 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    # ── [기존] macro_stress_score ─────────────────────────────
    .withColumn(
        "macro_stress_score",
        F.when(
            col("DGS10").isNotNull() &
            col("stagnation_pressure").isNotNull() &
            col("usd_krw_pct").isNotNull(),
            F.round(
                col("DGS10") + col("stagnation_pressure") + col("usd_krw_pct"),
                4
            )
        ).otherwise(F.lit(None).cast("double"))
    )

    # ── [신규] fear_composite ───────────────────────────────
    # 복합 공포 지수: 옵션 IV + 환율변화율 + 금리차변화
    # 세 시장의 공포 신호를 하나로 합산 (값이 클수록 공포)
    .withColumn(
        "fear_composite",
        F.when(
            col("avg_iv").isNotNull() &
            col("usd_krw_pct").isNotNull() &
            col("yield_spread_change").isNotNull(),
            F.round(
                col("avg_iv")
                + F.abs(col("usd_krw_pct")) * 10   # 환율변화 스케일 조정
                - col("yield_spread_change") * 5,   # 금리차 역전 심화 시 가산
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )

    # ── [신규] semi_risk_signal ────────────────────────────
    # 수출 감소(export_momentum=1) AND 변동성 고조 동시 발생
    # 펀더멘털(수출) + 테크니컬(변동성) 디버전스
    .withColumn(
        "semi_risk_signal",
        F.when(
            (col("export_momentum") == 1) &
            (col("NVDA_volatility_5d").isNotNull()) &
            (col("NVDA_volatility_5d") > 0.05),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    # ── [신규] korea_sensitivity ───────────────────────────
    # 환율변화율 × 옵션 IV 상호작용 (둘 다 스파이크 시 증폭)
    .withColumn(
        "korea_sensitivity",
        F.when(
            col("usd_krw_pct").isNotNull() & col("avg_iv").isNotNull(),
            F.round(F.abs(col("usd_krw_pct")) * col("avg_iv") / 10, 4)
        ).otherwise(F.lit(None).cast("double"))
    )

    # ── [신규] global_risk_regime ──────────────────────────
    # 다중 신호 기반 위험 국면 분류
    # 0 = 저위험, 1 = 중위험, 2 = 고위험
    .withColumn("_sig_fx",    F.when(col("risk_off_flag") == 1, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("_sig_iv",    F.when(col("iv_surge_flag") == 1, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("_sig_yield", F.when(col("yield_spread") < 0, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("_sig_semi",  F.when(col("export_momentum") == 1, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("_sig_total", col("_sig_fx") + col("_sig_iv") + col("_sig_yield") + col("_sig_semi"))
    .withColumn(
        "global_risk_regime",
        F.when(col("_sig_total") >= 3, F.lit(2))  # 3개 이상 → 고위험
         .when(col("_sig_total") >= 1, F.lit(1))  # 1~2개 → 중위험
         .otherwise(F.lit(0))                      # 0개 → 저위험
    )
    .drop("_sig_fx", "_sig_iv", "_sig_yield", "_sig_semi", "_sig_total")

    # ── 파티션 컬럼 ──────────────────────────────────────
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
)

print("✅ Gold 전용 파생열 생성 완료")
display(
    gold_featured_df
    .select("date", "fear_composite", "semi_risk_signal",
            "korea_sensitivity", "global_risk_regime",
            "risk_off_composite", "macro_stress_score")
    .orderBy("date")
    .limit(10)
)

# 7. 타겟 변수 생성 (is_high_risk)

ML 모델의 **정답지** 를 만드는 단계입니다.

### 기준
`NVDA_volatility_5d` 가 전체 데이터의 **상위 25%** 를 넘으면
하방 리스크가 높은 날로 판단하여 `is_high_risk = 1` 로 라벨링합니다.

### 쉬운 설명
> 1년치 변동성 데이터를 줄 세웠을 때
> 상위 25% 안에 드는 날 = "위험한 날" (1)
> 나머지 75% = "비교적 안전한 날" (0)
>
> 이 0/1 라벨을 정답지로 삼아
> XGBoost 가 "어떤 매크로 상황에서 1이 나오는가" 를 학습합니다.


In [0]:
# NVDA_volatility_5d 의 상위 25% 기준값 계산
threshold = (
    gold_featured_df
    .approxQuantile("NVDA_volatility_5d", [0.75], 0.01)[0]
)

print(f"✅ 타겟 변수 기준값 (75th percentile): {threshold:.6f}")

gold_labeled_df = (
    gold_featured_df
    .withColumn(
        "is_high_risk",
        F.when(col("NVDA_volatility_5d") > threshold, F.lit(1))
         .otherwise(F.lit(0))
    )
)

# 라벨 분포 확인
label_dist = gold_labeled_df.groupBy("is_high_risk").count().collect()
total = gold_labeled_df.count()
print(f"\n=== 🎯 타겟 변수 분포 ===")
for row in sorted(label_dist, key=lambda x: x["is_high_risk"]):
    pct = row["count"] / total * 100
    label = "위험 🔴" if row["is_high_risk"] == 1 else "안전 🟢"
    print(f"  {label} (is_high_risk={row['is_high_risk']}): {row['count']}일 ({pct:.1f}%)")


# 8. 데이터 검증


In [0]:
# ── 날짜 범위 및 행수 확인 ──────────────────────────────────
date_range = gold_labeled_df.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.count("date").alias("영업일수")
).collect()[0]

print("=== 📅 날짜 범위 ===")
print(f"  시작: {date_range['start_date']}")
print(f"  종료: {date_range['end_date']}")
print(f"  영업일수: {date_range['영업일수']}일")
print(f"  전체 컬럼 수: {len(gold_labeled_df.columns)}개")

# ── 핵심 컬럼 null 비율 확인 ─────────────────────────────────
print("\n=== 🔍 핵심 컬럼 null 비율 ===")
total = gold_labeled_df.count()
check_cols = [
    "NVDA", "^SOX", "NVDA_volatility_5d",
    "DGS10", "yield_spread", "usd_krw",
    "export_usd", "avg_iv",
    "fear_composite", "semi_risk_signal",
    "korea_sensitivity", "global_risk_regime",
    "risk_off_composite", "macro_stress_score", "is_high_risk"
]
for c in check_cols:
    null_cnt = gold_labeled_df.filter(col(c).isNull()).count()
    status = "✅" if null_cnt == 0 else "⚠️"
    print(f"  {status} {c}: null {null_cnt}건 ({null_cnt/total*100:.1f}%)")

# ── global_risk_regime 분포 ───────────────────────────────────
print("\n=== 🌡️ 위험 국면 분포 (global_risk_regime) ===")
display(
    gold_labeled_df
    .groupBy("global_risk_regime")
    .count()
    .orderBy("global_risk_regime")
)

# ── 전체 컬럼 목록 출력 ──────────────────────────────────
print(f"\n=== 📋 Gold 컬럼 전체 목록 ===")
for i, c in enumerate(gold_labeled_df.columns, 1):
    print(f"  {i:2d}. {c}")

# 9. Gold 저장

- 포맷: **Parquet**
- 파티션: `year` / `month`
- 모드: `overwrite`

> Gold 는 Silver 세 개가 모두 완료된 후 실행합니다.
> Silver 재처리 시 Gold 도 함께 재실행해야 합니다.


In [0]:
(
    gold_labeled_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(GOLD_PATH)
)

saved_files = dbutils.fs.ls(GOLD_PATH)
print(f"✅ Gold 저장 완료")
print(f"   경로     : {GOLD_PATH}")
print(f"   파티션 수: {len(saved_files)}개")
for f in saved_files:
    print(f"   - {f.name}")


# 10. 저장 확인 (sanity check)


In [0]:
df_check = spark.read.parquet(GOLD_PATH)

print(f"✅ Gold 읽기 확인 — 행수: {df_check.count():,}")
print(f"   컬럼 수: {len(df_check.columns)}개")

# 샘플 출력 (핵심 컬럼만)
display(
    df_check
    .select(
        "date", "NVDA", "^SOX",
        "NVDA_volatility_5d", "DGS10", "yield_spread",
        "usd_krw", "risk_off_composite",
        "macro_stress_score", "is_high_risk"
    )
    .orderBy("date")
    .limit(10)
)


In [0]:
display(df_check    
        .select(
        "date", "NVDA", "^SOX",
        "NVDA_volatility_5d", "DGS10", "yield_spread",
        "usd_krw", "risk_off_composite",
        "macro_stress_score", "is_high_risk")
        .orderBy("date")
        )

   
# Gold 레이어 정리본 (Power BI 대시보드 용)

## 개요
| 항목 | 값 |
|---|---|
| 경로 | `abfss://feature@3dtteam1adls.dfs.core.windows.net/sense_macro` |
| 포맷 | Parquet (year/month 파티션) |
| 행수 | **246행** (한국 영업일 기준, 유니크) |
| 컬럼 | 47개 |
| 기간 | 2025-04-08 ~ 2026-04-09 |
| 기준 날짜 | `date` = `effective_kr_date` (한국 시장 반영일) |
| null | NVDA 1건(0.4%), avg_iv 15건(6.1%), 나머지 0건 |

---

## ⭐ Power BI 핵심 파생열 (8개)

### 1️⃣ `global_risk_regime` — 종합 위험 국면
| 구분 | 설명 |
|---|---|
| 값 | **0** (저위험, 181일) / **1** (중위험, 65일) |
| 산식 | FX급등 + IV급등 + 금리역전 + 수출감소 → 합산 ≥1이면 1 |
| null | 0건 |
| **PBI 활용** | 신호등 KPI (초록/노랑), 슬라이서 필터, 조건부 서식 배경색 |

### 2️⃣ `is_high_risk` — ML 타겟 (변동성 위험일)
| 구분 | 설명 |
|---|---|
| 값 | **0** (안전, 183일 74%) / **1** (위험, 63일 26%) |
| 산식 | NVDA 5일 변동성 상위 25% 초과 시 1 |
| null | 0건 |
| **PBI 활용** | 카드 KPI (위험일 건수), 날짜별 빨간 배경, 월별 위험 비율 막대 차트 |

### 3️⃣ `fear_composite` — 복합 공포 지수
| 구분 | 설명 |
|---|---|
| 산식 | `avg_iv` + |`usd_krw_pct`|×10 - `yield_spread_change`×5 |
| 의미 | 옵션 IV + 환율 급등 + 금리역전 심화 → 값이 클수록 공포 |
| null | 15건 (avg_iv 초기값 부재) |
| **PBI 활용** | 시계열 라인 차트 (공포 추세), 게이지 차트, NVDA 주가와 이중축 비교 |

### 4️⃣ `macro_stress_score` — 매크로 스트레스 점수
| 구분 | 설명 |
|---|---|
| 산식 | `DGS10` + `stagnation_pressure` + `usd_krw_pct` |
| 의미 | 금리 부담 + 경기침체 압력 + 환율 충격 합산 |
| null | 0건 |
| **PBI 활용** | 일별 스트레스 라인 차트, 월별 평균 박스플롯, 임계점 기준선 표시 |

### 5️⃣ `korea_sensitivity` — 한국 시장 민감도
| 구분 | 설명 |
|---|---|
| 산식 | |`usd_krw_pct`| × `avg_iv` / 10 |
| 의미 | 환율 변동 × 옵션 공포 상호작용 — 둘 다 스파이크 시 증폭 |
| null | 15건 |
| **PBI 활용** | 스컴론 차트 (usd_krw_pct vs avg_iv, 크기=korea_sensitivity) |

### 6️⃣ `risk_off_composite` — 위험회피 복합 신호
| 구분 | 설명 |
|---|---|
| 산식 | `risk_off_flag`=1 AND `yield_spread`<0 동시 발생 |
| 의미 | 달러 급등(≥1%) + 장단기 금리 역전 → 극단적 위험회피 |
| null | 0건 |
| **PBI 활용** | 알림 아이콘 (1 발생 시 빨간 경고), 카드 KPI |

### 7️⃣ `semi_risk_signal` — 반도체 리스크 신호
| 구분 | 설명 |
|---|---|
| 산식 | `export_momentum`=1 AND `NVDA_volatility_5d`>5% |
| 의미 | 수출 감소 + NVDA 변동성 고조 → 펀더멘털/테크니컬 디버전스 |
| null | 0건 |
| **PBI 활용** | 반도체 페이지 경고 아이콘, 수출+변동성 이중축 차트 |

### 8️⃣ `iv_surge_flag` — IV 급등 신호
| 구분 | 설명 |
|---|---|
| 산식 | 코스피200 콜 IV 전월비 +5 이상 시 1 |
| 의미 | 옵션 시장 공포 급등 (VIX 대용 스파이크) |
| null | 0건 |
| **PBI 활용** | 게이지 경고등, fear_composite와 조합 시각화 |

---

## 📊 Power BI 대시보드 구성 제안

### 페이지 1: 종합 리스크 모니터
| 위젥 | 시각화 | 사용 컬럼 |
|---|---|---|
| 상단 KPI | 카드 3개 | `global_risk_regime` 신호등, `is_high_risk` 위험일수, 최신 `macro_stress_score` |
| 중앙 | 시계열 라인 | `fear_composite` + `macro_stress_score` 이중축 |
| 하단 | 열지도 | `date` × `global_risk_regime` (0=초록/1=노랑) |

### 페이지 2: 반도체 섹터 심층
| 위젥 | 시각화 | 사용 컬럼 |
|---|---|---|
| 상단 | 이중축 라인 | NVDA 종가 + `SOX_volatility_5d` |
| 중앙 | 막대 차트 | `export_change_pct` 월별 변화 |
| 하단 | 스컴론 | `usd_krw_pct` vs `avg_iv` (size=`korea_sensitivity`) |

### 페이지 3: 경보 대시보드
| 위젥 | 시각화 | 사용 컬럼 |
|---|---|---|
| 상단 | 경보 카드 | `risk_off_composite`, `semi_risk_signal`, `iv_surge_flag` |
| 중앙 | 타임라인 | 경보 발생일 마커 + `fear_composite` 배경 |
| 하단 | 테이블 | 경보 발생일 상세 (날짜, 환율, IV, 금리) |

---

## 코드 연동 예시 (Power BI DAX)

```dax
// 신호등 색상
Risk Color = 
    SWITCH(
        SELECTEDVALUE(Gold[global_risk_regime]),
        0, "Green",
        1, "Orange",
        2, "Red",
        "Gray"
    )

// 월별 위험일 비율
Monthly Risk Rate = 
    DIVIDE(
        CALCULATE(COUNTROWS(Gold), Gold[is_high_risk] = 1),
        COUNTROWS(Gold)
    )
```

   
# 11. ML 피처 상관분석

### 목적
1. **누수 컬럼 제외**: `NVDA`, `NVDA_log_return`, `NVDA_volatility_gk`, `NVDA_volatility_5d` → 타겟(`is_high_risk`) 산출에 직접 사용
2. **다중공선성 진단**: |r| > 0.85 인 피처 쌍 식별 → 중복 제거 후보
3. **타겟 상관 Top-15**: `is_high_risk`와 상관계수 절대값 기준 상위 15개 피처 선정

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ── 누수 컬럼 제외 + 비피처 제외 ─────────────────────────────────
LEAKAGE_COLS = ["NVDA", "NVDA_log_return", "NVDA_volatility_gk", "NVDA_volatility_5d"]
NON_FEATURE  = ["date", "year", "month"]
TARGET       = "is_high_risk"

all_cols = gold_labeled_df.columns
feature_cols = [c for c in all_cols if c not in LEAKAGE_COLS + NON_FEATURE + [TARGET]]

print(f"✅ 분석 대상 피처: {len(feature_cols)}개")
print(f"   누수 제외: {LEAKAGE_COLS}")
print(f"   비피처 제외: {NON_FEATURE}")

# ── Pandas 변환 (백틱 컬럼명 처리: . 포함 컬럼 Spark 해석 방지) ────
select_expr = [F.col(f"`{c}`") for c in feature_cols + [TARGET]]
pdf = gold_labeled_df.select(select_expr).toPandas()

# ── 상관행렬 계산 ──────────────────────────────────────────────
corr = pdf[feature_cols].corr()

# ── 1) 히트맵 ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 16))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=90, fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlation Matrix (Leakage Excluded)", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# ── 2) 다중공선성: |r| > 0.85 피처 쌍 ──────────────────────────
threshold_mc = 0.85
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i+1, len(feature_cols)):
        r = corr.iloc[i, j]
        if abs(r) > threshold_mc:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], round(r, 4)))

high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print(f"\n=== 🚨 다중공선성 경고 (|r| > {threshold_mc}) : {len(high_corr_pairs)}쌍 ===")
for a, b, r in high_corr_pairs:
    print(f"  {a:25s} ↔ {b:25s}  r = {r:+.4f}")

# ── 3) 타겟 상관 Top-15 ───────────────────────────────────────────
target_corr = pdf[feature_cols].corrwith(pdf[TARGET]).abs().sort_values(ascending=False)

print(f"\n=== 🎯 타겟(is_high_risk) 상관 Top-15 ===")
for i, (feat, r) in enumerate(target_corr.head(15).items(), 1):
    bar = "█" * int(r * 40)
    print(f"  {i:2d}. {feat:25s}  |r| = {r:.4f}  {bar}")

# ── 4) 타겟 상관 시각화 ─────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 6))
top15 = target_corr.head(15)
colors = ['#e74c3c' if v > 0.3 else '#f39c12' if v > 0.15 else '#3498db' for v in top15.values]
ax2.barh(range(len(top15)-1, -1, -1), top15.values, color=colors)
ax2.set_yticks(range(len(top15)-1, -1, -1))
ax2.set_yticklabels(top15.index, fontsize=10)
ax2.set_xlabel("|Correlation| with is_high_risk", fontsize=11)
ax2.set_title("Top-15 Features by Target Correlation", fontsize=13)
ax2.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Strong (0.3)')
ax2.axvline(x=0.15, color='orange', linestyle='--', alpha=0.5, label='Moderate (0.15)')
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()

   
# 12. 최종 ML 피처 선정 (15개)

### 제거 규칙
| 제거 대상 | 사유 | 대체 피처 |
|---|---|---|
| `NVDA`, `NVDA_log_return`, `NVDA_volatility_gk`, `NVDA_volatility_5d` | **데이터 누수** — 타겟 산출에 직접 사용 | 제외 |
| `TSM`, `AMD`, `INTC`, `ASML`, `MU`, `WDC` | 한국주 포함 종목간 r=0.88\~0.99 | `^SOX` 지수로 대체 |
| `005930.KS`, `000660.KS` | 미국주와 r=0.89\~0.99 | `^SOX` 지수로 대체 |
| `SOX_log_return` | `SOX_volatility_5d`와 역할 중복 | `SOX_volatility_5d` 유지 |
| `usd_krw_change` | `usd_krw_pct`와 r=0.9998 | `usd_krw_pct` 유지 |
| `T10Y2Y` | `yield_spread`와 r=0.9996 | `yield_spread` 유지 |
| `DGS2`, `DFII10` | `DGS10`과 r=0.86\~0.93 | `DGS10` 유지 |
| `stagnation_pressure` | = DGS10 + BAMLH0A0HYM2 (입력 합산) | 각각 유지 |
| `fear_composite` | `avg_iv`와 r=0.92 (파생) | `avg_iv` 유지 |
| `iv_change` | `iv_surge_flag`와 r=0.89 | `iv_surge_flag` 유지 |
| `call_volume`, `vol_change_pct` | 월별 FF 반복값, 정보량 낮음 | `call_oi` 유지 |
| `export_usd`, `export_momentum` | `export_change_pct`와 중복/파생 | `export_change_pct` 유지 |
| `risk_off_composite`, `macro_stress_score`, `semi_risk_signal` | 타겟 상관 낮음 (<0.10) | `global_risk_regime` 대체 |

In [0]:
# ========================================================
# 최종 15개 피처 정의
# ========================================================
FINAL_FEATURES = [
    # --- 주가 파생 (1) ---
    "SOX_volatility_5d",       # SOX 5일 변동성 (target |r|=0.46)
    # --- FX 환율 (3) ---
    "usd_krw",                 # USD/KRW 환율 수준
    "usd_krw_pct",             # 전일비 환율 변화율
    "risk_off_flag",           # 달러 급등 신호
    # --- FRED 금리 (4) ---
    "DGS10",                   # 10년물 국채 금리
    "DFF",                     # 연방기금 금리
    "BAMLH0A0HYM2",            # 하이일드 스프레드
    "yield_spread",            # 장단기 금리차 (DGS10-DGS2)
    "yield_spread_change",     # 금리차 일별 변화
    # --- 옵션 센티먼트 (3) ---
    "avg_iv",                  # VIX 대용 (IV 가중평균)
    "iv_surge_flag",           # IV 급등 신호
    "call_oi",                 # 콜옵션 미결제약정
    # --- 반도체 수출 (1) ---
    "export_change_pct",       # 수출 월별 변화율
    # --- Gold 파생 (2) ---
    "korea_sensitivity",       # |usd_krw_pct| x avg_iv / 10
    "global_risk_regime",      # 복합 위험 국면 (0/1/2)
]

TARGET = "is_high_risk"

# ========================================================
# 검증 1: 최종 피처 간 상관행렬
# ========================================================
pdf_final = pdf[FINAL_FEATURES + [TARGET]].dropna()
corr_final = pdf_final[FINAL_FEATURES].corr()

# |r| > 0.85 잔존 여부
high_pairs = []
for i in range(len(FINAL_FEATURES)):
    for j in range(i+1, len(FINAL_FEATURES)):
        r = corr_final.iloc[i, j]
        if abs(r) > 0.85:
            high_pairs.append((FINAL_FEATURES[i], FINAL_FEATURES[j], round(r, 4)))

if high_pairs:
    print(f"⚠️ 잔존 다중공선성 (|r|>0.85): {len(high_pairs)}쌍")
    for a, b, r in sorted(high_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f"   {a:25s} ↔ {b:25s}  r={r:+.4f}")
else:
    print("✅ 다중공선성 없음 — 모든 피처 쌍 |r| < 0.85")

# ========================================================
# 검증 2: 타겟 상관 순위
# ========================================================
target_corr_final = (
    pdf_final[FINAL_FEATURES]
    .corrwith(pdf_final[TARGET])
    .abs()
    .sort_values(ascending=False)
)

print(f"\n=== 🎯 최종 15개 피처 → 타겟 상관 ===")
print(f"   사용 행수: {len(pdf_final)}행 (null 제거 후)")
for i, (feat, r) in enumerate(target_corr_final.items(), 1):
    tier = "🔴 Strong" if r > 0.25 else "🟠 Moderate" if r > 0.12 else "🔵 Weak"
    bar = "█" * int(r * 50)
    print(f"  {i:2d}. {feat:25s}  |r|={r:.4f}  {bar}  {tier}")

# ========================================================
# 검증 3: 시각화
# ========================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# 좌: 상관 히트맵
im = axes[0].imshow(corr_final.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axes[0].set_xticks(range(len(FINAL_FEATURES)))
axes[0].set_yticks(range(len(FINAL_FEATURES)))
axes[0].set_xticklabels(FINAL_FEATURES, rotation=90, fontsize=8)
axes[0].set_yticklabels(FINAL_FEATURES, fontsize=8)
fig.colorbar(im, ax=axes[0], shrink=0.8)
axes[0].set_title("Final 15 Features \u2014 Correlation Matrix", fontsize=12)

# 우: 타겟 상관 바
colors = ['#e74c3c' if v > 0.25 else '#f39c12' if v > 0.12 else '#3498db'
          for v in target_corr_final.values]
axes[1].barh(range(len(target_corr_final)-1, -1, -1), target_corr_final.values, color=colors)
axes[1].set_yticks(range(len(target_corr_final)-1, -1, -1))
axes[1].set_yticklabels(target_corr_final.index, fontsize=9)
axes[1].set_xlabel("|r| with is_high_risk", fontsize=11)
axes[1].set_title("Target Correlation (Final 15)", fontsize=12)
axes[1].axvline(x=0.25, color='red', linestyle='--', alpha=0.4, label='Strong')
axes[1].axvline(x=0.12, color='orange', linestyle='--', alpha=0.4, label='Moderate')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n✅ 최종 피처 {len(FINAL_FEATURES)}개 + 타겟 1개 확정")
print(f"   사용 가능 행수: {len(pdf_final)}행")
print(f"   피처 목록: {FINAL_FEATURES}")